# 21 - Cost, Latency, and Economics

## Scenario: Implementing Token Budgets

If an agent gets confused, it can enter an infinite loop. If it iterates 500 times, passing a massive log file back and forth, you could be billed $50 for a single user query!

In production, you MUST implement **Token Budgets** or iteration caps. In this notebook, we'll build a wrapper that tracks usage and forcefully terminates the agent if it exceeds the allowed budget.

In [1]:
# 1. Initialization and Mock Fallback
import os
import sys

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("✅ Using real OpenAI API.")
else:
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    sys.path.append(os.path.abspath("../../.."))
    try:
        from awsome_agents.mock_openai import MockOpenAI
        client = MockOpenAI()
    except ImportError:
        print("Failed to import MockOpenAI. Ensure you are running from the repository root.")

⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. Tracking Token Usage

Every OpenAI API response includes a `usage` object. We will simulate tracking this across a loop.

In [2]:
class TokenBudgetTracker:
    def __init__(self, max_tokens: int):
        self.max_tokens = max_tokens
        self.current_tokens = 0
        
    def add_usage(self, tokens: int):
        self.current_tokens += tokens
        print(f"📊 [Budget] Consumed {tokens} tokens. Total: {self.current_tokens}/{self.max_tokens}")
        if self.current_tokens > self.max_tokens:
            raise RuntimeError(f"🚨 Token Budget Exceeded! Limit was {self.max_tokens}.")

# Initialize a strict budget
budget = TokenBudgetTracker(max_tokens=1000)

def simulate_agent_step(budget_tracker: TokenBudgetTracker):
    # Simulate an API call that consumes a random amount of tokens
    print("🧠 [Agent] Reasoning about the problem...")
    
    # In reality, this comes from `completion.usage.total_tokens`
    # We will simulate consuming 400 tokens per step
    tokens_consumed = 400
    budget_tracker.add_usage(tokens_consumed)
    return "Action taken."


## 2. Hitting the Circuit Breaker

In [3]:
print("Starting agent loop...\n")

try:
    # The agent gets stuck in a loop!
    for step in range(5):
        print(f"--- Step {step + 1} ---")
        simulate_agent_step(budget)
except RuntimeError as e:
    print(f"\n🛑 CIRCUIT BREAKER TRIGGERED: {e}")
    print("Escalating to human instead of burning more money.")


Starting agent loop...

--- Step 1 ---
🧠 [Agent] Reasoning about the problem...
📊 [Budget] Consumed 400 tokens. Total: 400/1000
--- Step 2 ---
🧠 [Agent] Reasoning about the problem...
📊 [Budget] Consumed 400 tokens. Total: 800/1000
--- Step 3 ---
🧠 [Agent] Reasoning about the problem...
📊 [Budget] Consumed 400 tokens. Total: 1200/1000

🛑 CIRCUIT BREAKER TRIGGERED: 🚨 Token Budget Exceeded! Limit was 1000.
Escalating to human instead of burning more money.


## Checkpoint

**1. Why is a Token Budget critical for Agentic systems?**
- A) It makes the agent smarter.
- B) Agents can autonomously invoke tools and loop indefinitely. A budget acts as a financial circuit breaker to prevent infinite loops from draining your API funds.
- C) It allows the agent to run locally without internet.
- D) It bypasses rate limits.
